<a href="https://colab.research.google.com/github/sourabhverma1302/AI-ML-Journey/blob/main/House_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [37]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

# ── Load data ─────────────────────────────────────────────
df = pd.read_csv('./data.csv')

# ── 1. Basic cleaning ─────────────────────────────────────
# Remove invalid target values
df = df[df['price'] > 0]

# Drop useless / leakage-prone columns
df = df.drop(columns=['street','country','date'])

# ── 2. Train-test split FIRST (very important) ────────────
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# ── 3. Safer target encoding (SMOOTHED) ───────────────────
global_mean = train_df['price'].mean()
smooth = 10

# --- City encoding ---
city_avg = train_df.groupby('city')['price'].mean()
city_count = train_df['city'].value_counts()

city_enc_map = (city_avg * city_count + global_mean * smooth) / (city_count + smooth)

train_df['city_enc'] = train_df['city'].map(city_enc_map)
test_df['city_enc']  = test_df['city'].map(city_enc_map).fillna(global_mean)

# --- Zip encoding ---
zip_avg = train_df.groupby('statezip')['price'].mean()
zip_count = train_df['statezip'].value_counts()

zip_enc_map = (zip_avg * zip_count + global_mean * smooth) / (zip_count + smooth)

train_df['zip_enc'] = train_df['statezip'].map(zip_enc_map)
test_df['zip_enc']  = test_df['statezip'].map(zip_enc_map).fillna(global_mean)

# ── 4. Define features ────────────────────────────────────
features = [
    'sqft_living','sqft_above','sqft_basement','sqft_lot',
    'bedrooms','bathrooms','floors','waterfront',
    'view','condition','city_enc','zip_enc'
]

X_train = train_df[features]
y_train = train_df['price']

X_test  = test_df[features]
y_test  = test_df['price']

# ── 5. Random Forest (REGULARIZED to avoid overfitting) ───
model = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# ── 6. Predictions ────────────────────────────────────────
y_train_pred = model.predict(X_train)
y_test_pred  = model.predict(X_test)

# ── 7. Evaluation ─────────────────────────────────────────
print(f"Train R²: {r2_score(y_train, y_train_pred):.3f}")
print(f"Test  R²: {r2_score(y_test,  y_test_pred):.3f}")
print(f"MAE:      ${mean_absolute_error(y_test, y_test_pred):,.0f}")

# ── 8. Feature Importance ─────────────────────────────────
importances = pd.Series(model.feature_importances_, index=features)
print("\nFeature importance:")
print(importances.sort_values(ascending=False).round(3))

# ── 9. Sample Predictions ─────────────────────────────────
results = pd.DataFrame({
    'actual':    y_test.values[:8],
    'predicted': y_test_pred[:8].round(0),
    'error':     (y_test_pred[:8] - y_test.values[:8]).round(0)
})

print("\nSample predictions:")
print(results.to_string(index=False))

Train R²: 0.437
Test  R²: 0.704
MAE:      $105,708

Feature importance:
sqft_living      0.282
zip_enc          0.221
sqft_above       0.150
city_enc         0.099
bathrooms        0.070
sqft_lot         0.063
sqft_basement    0.041
view             0.035
bedrooms         0.014
condition        0.012
floors           0.010
waterfront       0.004
dtype: float64

Sample predictions:
   actual  predicted    error
1225000.0  1247714.0  22714.0
 496752.0   535448.0  38696.0
 612500.0   563035.0 -49465.0
 265000.0   261289.0  -3711.0
 615000.0   603251.0 -11749.0
 432000.0   475891.0  43891.0
 305000.0   414300.0 109300.0
 405000.0   420629.0  15629.0
